# NYC Taxi · Payment Type · Spark MLlib

# NYC Taxi · Payment Type

**Тема:** классификация типа оплаты поездки по признакам самой поездки. Используются только поля, которые не являются итогом расчёта: провайдер, число пассажиров, расстояние/длительность, время, зоны посадки/высадки, rate code и store-and-forward flag.

- **Реальные данные:** New York City Taxi and Limousine Commission (TLC), Yellow Taxi Trip Records, январь–март 2025 года, Parquet.
- **Масштаб:** несколько месяцев поездок, миллионы строк; точный объём после проверки качества будет записан в `report.json`.
- **Leakage control:** не используем fare, tip, total amount и другие платежные суммы, поскольку это утечка информации об оплате.
- **Модели:** Spark Logistic Regression на полной train-выборке и Random Forest на ограниченной train-подвыборке; класс-баланс и метрики считаются через агрегаты Spark.

**Источники:** [официальный TLC Trip Record Data](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page), [официальный словарь полей Yellow Taxi](https://www.nyc.gov/assets/tlc/downloads/pdf/data_dictionary_trip_records_yellow.pdf).

Коды `payment_type` по словарю TLC: 0 = Flex Fare, 1 = Credit card, 2 = Cash, 3 = No charge, 4 = Dispute, 5 = Unknown, 6 = Voided trip. Это целевая переменная с сильным дисбалансом, поэтому смотрим на macro-F1 и recall редких классов, а не только accuracy.

## 0. Environment and Spark session

Все тяжёлые таблицы живут в Spark. Драйвер не использует `toPandas()` и не загружает полную выборку в локальную память. Shared volume `/opt/spark/data` должен быть смонтирован одинаковым путём в `ml` и `spark-worker` контейнерах.

In [ ]:
TASK = "taxi_payment"
APP_NAME = "NYC Taxi · Payment Type · Spark MLlib"

import os, json, math, shutil, zipfile, warnings
from pathlib import Path
from datetime import datetime

import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import StructType, StructField, IntegerType
from pyspark import StorageLevel
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator

warnings.filterwarnings("ignore")
SEED = 42

def find_project_root(start=None):
    p = Path(start or Path.cwd()).resolve()
    for parent in [p, *p.parents]:
        if (parent / "requirements.txt").exists() and (parent / "notebooks").exists():
            return parent
    # Container deployments typically mount repository at /app.
    if Path("/app/requirements.txt").exists() and Path("/app/notebooks").exists():
        return Path("/app")
    raise RuntimeError("Корень проекта не найден: ожидаются requirements.txt и notebooks/.")

PROJECT = find_project_root()
DATA_ROOT = Path(os.environ.get("SPARK_DATA_ROOT", "/opt/spark/data"))
DATA_ROOT.mkdir(parents=True, exist_ok=True)
MODEL_DIR = PROJECT / "models" / "classification" / TASK
ART_DIR = PROJECT / "artifacts" / "classification" / TASK
MODEL_DIR.mkdir(parents=True, exist_ok=True)
ART_DIR.mkdir(parents=True, exist_ok=True)

master_url = os.environ.get("SPARK_MASTER_URL", "local[2]")
builder = (SparkSession.builder.appName(APP_NAME)
    .master(master_url)
    .config("spark.driver.host", os.environ.get("SPARK_DRIVER_HOST", "ml-playground"))
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.driver.memory", "4g")
    .config("spark.executor.memory", "8g")
    .config("spark.executor.cores", "5")
    .config("spark.cores.max", "5")
    .config("spark.driver.maxResultSize", "512m")
    .config("spark.sql.shuffle.partitions", "20")
    .config("spark.default.parallelism", "20")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "file:///opt/spark/events"))
spark = builder.getOrCreate()
spark.sparkContext.setLogLevel("WARN")

print("PROJECT   :", PROJECT)
print("DATA_ROOT :", DATA_ROOT)
print("MODEL_DIR :", MODEL_DIR)
print("ART_DIR   :", ART_DIR)
print("SPARK     :", spark.version, "| master =", spark.sparkContext.master)


## 1. Загрузка и подготовка реального датасета

Исходные данные скачиваются с официального источника; подготовленная таблица сохраняется в Parquet в Docker shared volume. Чтобы принудительно пересобрать датасет, включите `REBUILD_DATA = True` в ячейке подготовки.

In [ ]:
TASK = "taxi_payment"
APP_NAME = "ML Explorer · NYC Taxi Payment Type · 2025 Q1"
DATASET_DIR = DATA_ROOT / TASK
RAW_DIR = DATASET_DIR / "raw"
CLEAN_DIR = DATASET_DIR / "processed"
RAW_DIR.mkdir(parents=True, exist_ok=True)
MONTHS = ["2025-01", "2025-02", "2025-03"]
TLC_BASE = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_{month}.parquet"
REBUILD_DATA = False
READY_MARKER = DATASET_DIR / "processed.ready"
if REBUILD_DATA:
    if CLEAN_DIR.exists(): shutil.rmtree(CLEAN_DIR)
    READY_MARKER.unlink(missing_ok=True)

if not READY_MARKER.exists():
    if CLEAN_DIR.exists(): shutil.rmtree(CLEAN_DIR)
    CLEAN_DIR.mkdir(parents=True, exist_ok=True)
    for month in MONTHS:
        parquet_path = RAW_DIR / f"yellow_tripdata_{month}.parquet"
        if not parquet_path.exists():
            url = TLC_BASE.format(month=month)
            print(f"Downloading {month}: {url}")
            with requests.get(url, stream=True, timeout=(30, 180)) as response:
                response.raise_for_status()
                temp = parquet_path.with_suffix(".parquet.part")
                with open(temp, "wb") as f:
                    for chunk in response.iter_content(chunk_size=1024 * 1024):
                        if chunk: f.write(chunk)
                temp.replace(parquet_path)
        raw = spark.read.parquet(str(parquet_path))
        required = ["VendorID", "tpep_pickup_datetime", "tpep_dropoff_datetime", "passenger_count", "trip_distance", "RatecodeID", "store_and_fwd_flag", "PULocationID", "DOLocationID", "payment_type"]
        missing = [c for c in required if c not in raw.columns]
        if missing:
            raise RuntimeError(f"TLC file {month} missing columns: {missing}")
        prepared = raw.select(*required)
        prepared = (prepared
            .withColumn("pickup_ts", F.col("tpep_pickup_datetime").cast("timestamp"))
            .withColumn("dropoff_ts", F.col("tpep_dropoff_datetime").cast("timestamp"))
            .withColumn("passenger_count", F.col("passenger_count").cast("double"))
            .withColumn("trip_distance", F.col("trip_distance").cast("double"))
            .withColumn("VendorID", F.col("VendorID").cast("double"))
            .withColumn("RatecodeID", F.col("RatecodeID").cast("double"))
            .withColumn("PULocationID", F.col("PULocationID").cast("double"))
            .withColumn("DOLocationID", F.col("DOLocationID").cast("double"))
            .withColumn("payment_type", F.col("payment_type").cast("int"))
            .withColumn("trip_duration_min", (F.col("dropoff_ts").cast("long") - F.col("pickup_ts").cast("long")) / 60.0)
            .withColumn("pickup_hour", F.hour("pickup_ts").cast("double"))
            .withColumn("pickup_dow", F.dayofweek("pickup_ts").cast("double"))
            .withColumn("pickup_month", F.month("pickup_ts").cast("double"))
            .withColumn("store_and_fwd_flag", F.coalesce(F.col("store_and_fwd_flag").cast("string"), F.lit("U")))
            .filter(F.col("payment_type").isin(0, 1, 2, 3, 4, 5, 6))
            .filter(F.col("pickup_ts").isNotNull() & F.col("dropoff_ts").isNotNull())
            .filter(F.col("trip_duration_min").between(1.0, 240.0))
            .filter(F.col("trip_distance").between(0.1, 150.0))
            .filter(F.col("passenger_count").isNull() | F.col("passenger_count").between(0.0, 8.0))
            .select(
                F.col("payment_type").cast("string").alias("payment_type_raw"),
                F.coalesce(F.col("VendorID"), F.lit(-1.0)).alias("vendor_id"),
                F.coalesce(F.col("passenger_count"), F.lit(0.0)).alias("passenger_count"),
                F.col("trip_distance").alias("trip_distance_miles"),
                F.coalesce(F.col("RatecodeID"), F.lit(-1.0)).alias("rate_code"),
                F.coalesce(F.col("PULocationID"), F.lit(-1.0)).alias("pickup_zone"),
                F.coalesce(F.col("DOLocationID"), F.lit(-1.0)).alias("dropoff_zone"),
                F.col("pickup_hour"), F.col("pickup_dow"), F.col("pickup_month"), F.col("trip_duration_min"),
                F.col("store_and_fwd_flag")
            ))
        prepared.write.mode("append").parquet(str(CLEAN_DIR))
        print(f"{month} processed; rows written to shared Parquet directory.")
    READY_MARKER.write_text(datetime.now().isoformat(), encoding="utf-8")
else:
    print("Found completed processed Parquet; skip downloads. Set REBUILD_DATA=True to rebuild.")

print("Prepared data:", CLEAN_DIR)


## 2. EDA и контроль распределения классов

Счётчики вычисляются Spark-агрегациями; в память Python передаются только несколько чисел/классов.

## 3. Train/test split и модели Spark MLlib

Результаты сравниваются с baseline большинства. Logistic Regression обучается на всей train-части. Random Forest обучается на train-подвыборке с лимитом строк для предсказуемого времени и памяти; test для обеих моделей полный и общий.

In [ ]:
# Метрики считаются из агрегированной confusion matrix — ни одна большая выборка
# не переносится в pandas/NumPy целиком. Spark возвращает только K×K счётчики.
def metrics_from_cm(cm):
    cm = np.asarray(cm, dtype=np.int64)
    total = int(cm.sum())
    if total == 0:
        raise ValueError("Пустая тестовая выборка")
    true_support = cm.sum(axis=1).astype(float)
    pred_support = cm.sum(axis=0).astype(float)
    tp = np.diag(cm).astype(float)
    precision = np.divide(tp, pred_support, out=np.zeros_like(tp), where=pred_support > 0)
    recall = np.divide(tp, true_support, out=np.zeros_like(tp), where=true_support > 0)
    f1 = np.divide(2 * precision * recall, precision + recall, out=np.zeros_like(tp), where=(precision + recall) > 0)
    accuracy = float(tp.sum() / total)
    weighted_f1 = float(np.average(f1, weights=true_support)) if true_support.sum() else 0.0
    numerator = float(total * tp.sum() - np.dot(true_support, pred_support))
    denom_left = float(total**2 - np.dot(pred_support, pred_support))
    denom_right = float(total**2 - np.dot(true_support, true_support))
    denom = math.sqrt(max(denom_left, 0.0) * max(denom_right, 0.0))
    mcc = numerator / denom if denom > 0 else 0.0
    expected = float(np.dot(true_support, pred_support) / (total**2))
    kappa = (accuracy - expected) / (1.0 - expected) if expected < 1.0 else 0.0
    metrics = {
        "accuracy": accuracy,
        "balanced_accuracy": float(recall.mean()),
        "precision_macro": float(precision.mean()),
        "recall_macro": float(recall.mean()),
        "f1_macro": float(f1.mean()),
        "f1_weighted": weighted_f1,
        "mcc": float(mcc),
        "kappa": float(kappa),
    }
    per_class = {}
    for i, label in enumerate(CLASSES):
        per_class[label] = {
            "precision": float(precision[i]),
            "recall": float(recall[i]),
            "f1": float(f1[i]),
            "support": int(true_support[i]),
            "is_rare": label == RARE_CLASS,
        }
    return metrics, per_class

def confusion_from_predictions(predictions):
    matrix = np.zeros((len(CLASSES), len(CLASSES)), dtype=np.int64)
    rows = predictions.groupBy(F.col("label").cast("int").alias("actual"), F.col("prediction").cast("int").alias("predicted")).count().collect()
    for row in rows:
        a, p = int(row["actual"]), int(row["predicted"])
        if 0 <= a < len(CLASSES) and 0 <= p < len(CLASSES):
            matrix[a, p] = int(row["count"])
    return matrix

def baseline_confusion(test_counts, majority_index):
    matrix = np.zeros((len(CLASSES), len(CLASSES)), dtype=np.int64)
    for actual_idx, count in enumerate(test_counts):
        matrix[actual_idx, majority_index] = int(count)
    return matrix

def json_safe(value):
    if isinstance(value, dict):
        return {str(k): json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(v) for v in value]
    if isinstance(value, np.ndarray):
        return json_safe(value.tolist())
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating, float)):
        val = float(value)
        return val if math.isfinite(val) else None
    if isinstance(value, (np.bool_,)):
        return bool(value)
    return value

def evaluate_model(name, model, eval_df, train_rows, model_path):
    print(f"Scoring {name} ...")
    scored = model.transform(eval_df).select("label", "prediction", "rawPrediction").persist(StorageLevel.MEMORY_AND_DISK)
    cm = confusion_from_predictions(scored)
    metrics, per_class = metrics_from_cm(cm)
    if len(CLASSES) == 2:
        for metric_name, output_name in [("areaUnderROC", "roc_auc"), ("areaUnderPR", "pr_auc")]:
            try:
                evaluator = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="rawPrediction", metricName=metric_name)
                metrics[output_name] = float(evaluator.evaluate(scored))
            except Exception as exc:
                print(f"{output_name} skipped: {exc}")
                metrics[output_name] = None
    scored.unpersist()
    return {
        "name": name,
        "is_best": False,
        "is_naive": False,
        "train_rows": int(train_rows),
        "model_path": str(model_path.relative_to(PROJECT)),
        "metrics": metrics,
        "confusion_matrix": cm.tolist(),
        "per_class": per_class,
        "per_class_recall": {label: per_class[label]["recall"] for label in CLASSES},
    }

# Простая baseline-модель: всегда выбирает самый частый класс в TRAIN.
# Её confusion matrix строится из агрегированных counts теста.
def make_baseline(train_counts, test_counts):
    majority_idx = int(np.argmax(train_counts))
    cm = baseline_confusion(test_counts, majority_idx)
    metrics, per_class = metrics_from_cm(cm)
    if len(CLASSES) == 2:
        metrics["roc_auc"] = None
        metrics["pr_auc"] = None
    return {
        "name": "Baseline · majority class",
        "is_best": False,
        "is_naive": True,
        "train_rows": 0,
        "metrics": metrics,
        "confusion_matrix": cm.tolist(),
        "per_class": per_class,
        "per_class_recall": {label: per_class[label]["recall"] for label in CLASSES},
    }

def build_feature_stages(numeric_cols, categorical_cols):
    stages = []
    indexed_cols, encoded_cols = [], []
    if categorical_cols:
        indexed_cols = [f"{c}_idx" for c in categorical_cols]
        encoded_cols = [f"{c}_ohe" for c in categorical_cols]
        stages.append(StringIndexer(inputCols=categorical_cols, outputCols=indexed_cols, handleInvalid="keep"))
        stages.append(OneHotEncoder(inputCols=indexed_cols, outputCols=encoded_cols, handleInvalid="keep", dropLast=False))
    assembler_inputs = list(numeric_cols) + encoded_cols
    stages.append(VectorAssembler(inputCols=assembler_inputs, outputCol="features", handleInvalid="keep"))
    return stages

def create_pipeline(classifier, numeric_cols, categorical_cols):
    return Pipeline(stages=build_feature_stages(numeric_cols, categorical_cols) + [classifier])

# В выборках несколько миллионов строк, поэтому не используем toPandas() по raw data.


In [ ]:
TASK = "taxi_payment"
APP_NAME = "ML Explorer · NYC Taxi Payment Type · 2025 Q1"
PAYMENT_NAMES = {"0": "Flex Fare", "1": "Credit card", "2": "Cash", "3": "No charge", "4": "Dispute", "5": "Unknown", "6": "Voided trip"}
SOURCE = "NYC Taxi & Limousine Commission (TLC) · Yellow Taxi Trip Records"
SOURCE_URL = "https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page"
TITLE = "NYC Taxi · Payment Type"
SUBTITLE = "Предсказание типа оплаты по характеристикам поездки · январь–март 2025"
DESCRIPTION = "Суммы оплаты, чаевые и итоговый тариф намеренно исключены из признаков: они напрямую связаны с целевой переменной и создавали бы target leakage."
NUMERIC_COLS = ["passenger_count", "trip_distance_miles", "trip_duration_min"]
CATEGORICAL_COLS = ["vendor_id", "rate_code", "pickup_zone", "dropoff_zone", "pickup_hour", "pickup_dow", "pickup_month", "store_and_fwd_flag"]
RF_TRAIN_CAP = 350_000

df_raw = spark.read.parquet(str(CLEAN_DIR)).persist(StorageLevel.MEMORY_AND_DISK)
n_samples = df_raw.count()
if n_samples < 100_000:
    raise RuntimeError(f"После очистки осталось подозрительно мало поездок: {n_samples:,}")
label_indexer_model = StringIndexer(inputCol="payment_type_raw", outputCol="label", stringOrderType="frequencyDesc", handleInvalid="skip").fit(df_raw)
label_strings = label_indexer_model.labels
CLASSES = [PAYMENT_NAMES.get(str(int(float(code))), f"Payment code {code}") for code in label_strings]
df = label_indexer_model.transform(df_raw).select("label", *NUMERIC_COLS, *CATEGORICAL_COLS).persist(StorageLevel.MEMORY_AND_DISK)
# Count before dropping old raw frame; label indices are stable and match CLASSES.
n_samples = df.count()
df_raw.unpersist()
counts_all_rows = df.groupBy("label").count().collect()
class_counts_idx = [0] * len(CLASSES)
for row in counts_all_rows: class_counts_idx[int(float(row["label"]))] = int(row["count"])
RARE_CLASS = CLASSES[int(np.argmin(class_counts_idx))]

print(f"Rows after quality filters: {n_samples:,} | input features: {len(NUMERIC_COLS) + len(CATEGORICAL_COLS)} | classes: {len(CLASSES)}")
for label, count in zip(CLASSES, class_counts_idx): print(f"  {label}: {count:,} ({count/n_samples:.3%})")
fig, ax = plt.subplots(figsize=(10, 4.5))
ax.bar(CLASSES, class_counts_idx, color=["#fb7185" if label == RARE_CLASS else "#6366f1" for label in CLASSES])
ax.set_title("NYC Yellow Taxi · payment type distribution")
ax.set_ylabel("Trips"); ax.tick_params(axis="x", rotation=20)
plt.tight_layout(); plt.show()

train, test = df.randomSplit([0.80, 0.20], seed=SEED)
train = train.persist(StorageLevel.MEMORY_AND_DISK); test = test.persist(StorageLevel.MEMORY_AND_DISK)
n_train, n_test = train.count(), test.count()
df.unpersist()
train_counts = [0] * len(CLASSES); test_counts = [0] * len(CLASSES)
for row in train.groupBy("label").count().collect(): train_counts[int(float(row["label"]))] = int(row["count"])
for row in test.groupBy("label").count().collect(): test_counts[int(float(row["label"]))] = int(row["count"])

results = [make_baseline(train_counts, test_counts)]
fitted_models = {}
lr = LogisticRegression(labelCol="label", featuresCol="features", maxIter=25, regParam=0.05, elasticNetParam=0.0, standardization=True, family="multinomial")
lr_pipeline = create_pipeline(lr, NUMERIC_COLS, CATEGORICAL_COLS)
lr_path = MODEL_DIR / "taxi_payment_logistic_regression_spark"
print("Fit multinomial Logistic Regression on complete train split ...")
lr_model = lr_pipeline.fit(train)
lr_model.write().overwrite().save(str(lr_path))
fitted_models["Logistic Regression · full train"] = lr_model
results.append(evaluate_model("Logistic Regression · full train", lr_model, test, n_train, lr_path))

rf_fraction = min(1.0, RF_TRAIN_CAP / max(n_train, 1))
rf_train = train.sample(withReplacement=False, fraction=rf_fraction, seed=SEED + 1).persist(StorageLevel.MEMORY_AND_DISK)
rf_rows = rf_train.count()
rf = RandomForestClassifier(labelCol="label", featuresCol="features", numTrees=16, maxDepth=9, maxBins=32, featureSubsetStrategy="sqrt", seed=SEED)
rf_pipeline = create_pipeline(rf, NUMERIC_COLS, CATEGORICAL_COLS)
rf_path = MODEL_DIR / "taxi_payment_random_forest_spark"
print(f"Fit Random Forest on {rf_rows:,} Spark-sampled training rows ...")
rf_model = rf_pipeline.fit(rf_train)
rf_model.write().overwrite().save(str(rf_path))
fitted_models["Random Forest · Spark sample"] = rf_model
results.append(evaluate_model("Random Forest · Spark sample", rf_model, test, rf_rows, rf_path))
rf_train.unpersist()

best = max([r for r in results if not r.get("is_naive")], key=lambda r: r["metrics"].get("f1_macro", -1))
for result in results: result["is_best"] = result["name"] == best["name"]
class_distribution = {CLASSES[int(float(row["label"]))]: int(row["count"]) for row in counts_all_rows}
max_count, min_count = max(class_counts_idx), max(1, min(class_counts_idx))
meta = {
    "id": TASK, "title": TITLE, "subtitle": SUBTITLE, "description": DESCRIPTION,
    "source": SOURCE, "source_url": SOURCE_URL, "generated_at": datetime.now().isoformat(),
    "hero_metric": "f1_macro", "target_name": "payment_type", "classes": CLASSES,
    "dataset_stats": {"n_samples": int(n_samples), "n_features": len(NUMERIC_COLS) + len(CATEGORICAL_COLS), "n_classes": len(CLASSES), "n_train": int(n_train), "n_test": int(n_test)},
    "target_stats": {"class_distribution": class_distribution, "rare_class": RARE_CLASS, "imbalance_ratio": float(max_count / min_count), "rare_class_ratio": float(min_count / n_samples)},
}
report = {"meta": meta, "models": results}
report_path = ART_DIR / "report.json"
report_path.write_text(json.dumps(json_safe(report), ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")
pd.DataFrame([{ "model": r["name"], **r["metrics"], "train_rows": r.get("train_rows", 0) } for r in results]).to_csv(ART_DIR / "metrics_all_models.csv", index=False)
print("Best model by macro-F1:", best["name"], "| macro-F1 =", round(best["metrics"]["f1_macro"], 4))
print("Report:", report_path)

fig, ax = plt.subplots(figsize=(10, 4.5))
plot_rows = [r for r in results if r.get("metrics", {}).get("f1_macro") is not None]
ax.bar([r["name"] for r in plot_rows], [r["metrics"]["f1_macro"] for r in plot_rows], color=["#94a3b8" if r.get("is_naive") else ("#10b981" if r.get("is_best") else "#6366f1") for r in plot_rows])
ax.set_title("NYC Taxi · macro-F1 on the test split"); ax.set_ylim(0, 1); ax.tick_params(axis="x", rotation=15)
plt.tight_layout(); plt.show()


## 4. Сохранённые артефакты

Ноутбук создаёт Spark PipelineModel в `models/classification/<task>/` и `report.json` + `metrics_all_models.csv` в `artifacts/classification/<task>/`. Фронтенд читает `report.json`; перезапустите страницу после завершения ноутбука. Для повторного запуска Spark можно остановить ячейкой ниже.

In [ ]:
print("Model directory:", MODEL_DIR)
for item in sorted(MODEL_DIR.iterdir()):
    print(" ", item.name)
print("Artifact directory:", ART_DIR)
for item in sorted(ART_DIR.iterdir()):
    print(" ", item.name, f"({item.stat().st_size / 1024:.1f} KB)" if item.is_file() else "<directory>")

# Раскомментируйте, если закончена работа в текущей Spark-сессии.
# spark.stop()